# Cross-Currency Swaptions in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

A swaption gives its holder the right to enter an interest rate swap. In LUSID that's an
`InterestRateSwaption`, and the underlying `InterestRateSwap` lives inline on its `swap` field --
there's no separate instrument to create for the swap itself. It's "cross-currency" purely because
the underlying swap's two legs sit in different currencies: here a USD fixed leg against a
EUR-style floating leg.

## Valuation date constraint

The option's real expiry lives on its `exercise_date` field. Leave it unset and it quietly
defaults to the underlying swap's `start_date` -- but this notebook sets it explicitly, which
decouples the two dates: the option can expire on one day and the swap it delivers into can start
days later. 

`start_date` also has to come strictly before `exercise_date`. That's because it's the option's
own inception date, not a settlement date, so it can't just be copied from the underlying swap's
start.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format


SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

Under `SimpleStatic`, the swaption is reported at its own quoted mark. The swap's legs still
describe what the instrument actually is -- the fixed rate, the floating spread, the two
currencies -- but none of those numbers feed into the valuation, so there's no need for fixing
quotes on the floating leg either.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "CrossCurrencySwaptionDemo"
RECIPE    = "cross-currency-swaption-demo-recipe"
PORTFOLIO = "cross-currency-swaption-demo-book"

SWAPTION_ID = "DEMO-XCCY-SWPN-01"
DESC        = "Demo 1Y x 5Y USD/EUR Cross-Currency Swaption"

TRADE_DATE = d(2025, 3, 3)
EXPIRY     = d(2026, 3, 3)          # option expiry, ~1Y after trade date
SWAP_START = d(2026, 3, 5)          # underlying swap begins on exercise, T+2 settlement lag
SWAP_MATURITY = d(2031, 3, 5)       # 5Y swap tenor from the swap start
ASOF       = d(2025, 12, 15)        # strictly before EXPIRY

FIXED_CCY    = "USD"
FLOAT_CCY    = "EUR"
FIXED_RATE   = 0.0375                # 3.75%
FLOAT_SPREAD = 0.0010                # +10bp over the floating index
FIXED_DIRECTION = "Receive"          # the option holder receives fixed if exercised
FLOAT_DIRECTION = "Pay"
FREQUENCY  = "6M"
DAY_COUNT  = "Act360"
FLOAT_INDEX          = "ESTR"
FLOAT_FIXING_REF     = "EUR-ESTR"

EXERCISE_TYPE   = "European"
DELIVERY_METHOD = "Cash"

NOTIONAL = 1.0                      # per-unit contract; position quantity scales it

QUANTITY = 10_000_000.00
PRICE    = 0.018                    # quoted mark, in domCcy per unit notional
DENOM    = 1

print(f"{DESC}")
print(f"  expiry {EXPIRY:%Y-%m-%d}, into a swap {SWAP_START:%Y-%m-%d} -> {SWAP_MATURITY:%Y-%m-%d}")
print(f"  fixed   {FIXED_DIRECTION:<8} {FIXED_CCY} {FIXED_RATE:.2%}")
print(f"  float   {FLOAT_DIRECTION:<8} {FLOAT_CCY} {FLOAT_INDEX} + {FLOAT_SPREAD:.2%}")
print(f"  {QUANTITY:,.0f} notional at {PRICE} on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {FIXED_CCY}")

Demo 1Y x 5Y USD/EUR Cross-Currency Swaption
  expiry 2026-03-03, into a swap 2026-03-05 -> 2031-03-05
  fixed   Receive  USD 3.75%
  float   Pay      EUR ESTR + 0.10%
  10,000,000 notional at 0.018 on 2025-12-15
  market value = 10,000,000 x 0.018 / 1 = 180,000.00 USD


---
# 1. Instrument creation

The swaption's `swap` field carries the underlying `InterestRateSwap` inline, and it's the two
legs sitting in different currencies that makes this one cross-currency.

In [3]:
fixed_leg = m.FixedLeg(
    instrument_type="FixedLeg",
    start_date=SWAP_START,
    maturity_date=SWAP_MATURITY,
    notional=NOTIONAL,
    leg_definition=m.LegDefinition(
        rate_or_spread=FIXED_RATE,
        pay_receive=FIXED_DIRECTION,
        conventions=m.FlowConventions(
            currency=FIXED_CCY,
            payment_frequency=FREQUENCY,
            day_count_convention=DAY_COUNT,
            roll_convention=str(SWAP_START.day),
            payment_calendars=[], reset_calendars=[]),
        notional_exchange_type="None",
        stub_type="ShortBack"))

floating_leg = m.FloatingLeg(
    instrument_type="FloatingLeg",
    start_date=SWAP_START,
    maturity_date=SWAP_MATURITY,
    notional=NOTIONAL,
    leg_definition=m.LegDefinition(
        rate_or_spread=FLOAT_SPREAD,
        pay_receive=FLOAT_DIRECTION,
        conventions=m.FlowConventions(
            currency=FLOAT_CCY,
            payment_frequency=FREQUENCY,
            day_count_convention=DAY_COUNT,
            roll_convention=str(SWAP_START.day),
            payment_calendars=[], reset_calendars=[]),
        index_convention=m.IndexConvention(
            currency=FLOAT_CCY,
            payment_tenor="1D",
            fixing_reference=FLOAT_FIXING_REF,
            index_name=FLOAT_INDEX,
            publication_day_lag=0,
            day_count_convention=DAY_COUNT),
        reset_convention="InArrears",
        stub_type="ShortBack",
        notional_exchange_type="None"))

underlying_swap = m.InterestRateSwap(
    instrument_type="InterestRateSwap",
    start_date=SWAP_START,
    maturity_date=SWAP_MATURITY,
    legs=[fixed_leg, floating_leg])

swaption = m.InterestRateSwaption(
    instrument_type="InterestRateSwaption",
    start_date=TRADE_DATE,
    exercise_date=EXPIRY,
    pay_or_receive_fixed=FIXED_DIRECTION,
    delivery_method=DELIVERY_METHOD,
    exercise_type=EXERCISE_TYPE,
    dom_ccy=FIXED_CCY,
    swap=underlying_swap)

SWAPTION_LUID = upsert("swaption", DESC, SWAPTION_ID, swaption)
print(f"Cross-currency swaption : {SWAPTION_LUID}")

Cross-currency swaption : LUID_00003DFX


---
# 2. Recipe

With `SimpleStatic`, the position is reported at a quoted mark. The swap's two-currency legs
still describe what the instrument is, they just don't feed into the number under this model.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Cross-currency swaption, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="InterestRateSwaption")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: CrossCurrencySwaptionDemo/cross-currency-swaption-demo-recipe


---
# 3. Transaction types

A swaption's life ends at expiry -- there's no `MaturityEvent` for the option itself, just an
`ExpiryEvent`, which applies right up to and including the day it lapses or gets exercised into
the underlying swap. Until a transaction type is registered for that event, the `.transactions`
list on the matching `ApplicableInstrumentEvent` just stays empty.

Calling `get_transaction_template_specification("ExpiryEvent")` shows its `supportedInstrumentTypes`:
six option-like instruments sharing this one event -- `InterestRateSwaption`, `EquityOption`,
`ExchangeTradedOption`, `ToBeAnnouncedOption`, `BondOption`, `CdsOption` -- and none of the
template's fields vary by instrument type. LUSID works out the transaction type by stripping
`Event` off the event type's name, so what needs registering is the plain name `Expiry`, not
something instrument-qualified like `InterestRateSwaptionExpiry`. Register `Expiry` once and it
covers every instrument type in that list.

In [5]:
event_types_api = api(lusid.InstrumentEventTypesApi)
txn_config_api  = api(lusid.TransactionConfigurationApi)

spec = event_types_api.get_transaction_template_specification(instrument_event_type="ExpiryEvent")
print("ExpiryEvent applies to:", spec.supported_instrument_types)

txn_config_api.set_transaction_type(
    source="default", type="Expiry", scope="default",
    transaction_type_request=m.TransactionTypeRequest(
        aliases=[m.TransactionTypeAlias(
            type="Expiry", description="Option/swaption expiry -- closes the expired position",
            transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
        movements=[m.TransactionTypeMovement(
            movement_types="StockMovement", side="Side1", direction=-1)]))

print("Registered transaction type: Expiry (scope=default, source=default)")

ExpiryEvent applies to: ['InterestRateSwaption', 'EquityOption', 'ExchangeTradedOption', 'ToBeAnnouncedOption', 'BondOption', 'CdsOption']


Registered transaction type: Expiry (scope=default, source=default)


---
# 4. Portfolio and transactions

Striking a swaption doesn't move any principal, so `totalConsideration` here is zero.

`instrumentEventConfiguration`, set below via `recreate_portfolio(..., recipe=RECIPE)`, is what
lets section 6's events query return anything at all. It can only be set when the portfolio is
first created.

In [6]:
recreate_portfolio(PORTFOLIO, "Cross-Currency Swaption Demo Book", FIXED_CCY, d(2025, 1, 1),
                    recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-SWAPTION",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": SWAPTION_LUID},
        transaction_date=TRADE_DATE.isoformat(),
        settlement_date=TRADE_DATE.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=FIXED_CCY),
        source="default")])

display(transactions(PORTFOLIO, TRADE_DATE, TRADE_DATE))

Recreated CrossCurrencySwaptionDemo/cross-currency-swaption-demo-book


,date,type,luid,units,consideration
0,2025-03-03,Buy,LUID_00003DFX,"10,000,000.00",0.00


---
# 5. Valuation

Just one quote, at the swaption's own price, valued strictly before expiry.

In [7]:
upsert_price(SWAPTION_LUID, PRICE / DENOM, ASOF, FIXED_CCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, FIXED_CCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo 1Y x 5Y USD/EUR Cross-Currency Swaption,"10,000,000.00","180,000.00"


LUSID CleanPV 180,000.00  vs  quoted mark 180,000.00


---
# 6. Instrument events

The applicable event here is `ExpiryEvent`, fired off the swaption's own `exerciseDate` -- its
expiry, with no `MaturityEvent` involved for the option itself. The forecasted `Expiry`
transaction settles on that same expiry date, so the events window just needs to run a few days
past `EXPIRY` to catch it.

## 6a. Applicable events

In [8]:
events_api = api(lusid.InstrumentEventsApi)

EVENTS_WINDOW_END = EXPIRY + timedelta(days=5)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=TRADE_DATE.isoformat(),
        window_end=EVENTS_WINDOW_END.isoformat(),
        effective_at=EVENTS_WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

print(f"{len(applicable)} applicable event(s):\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

1 applicable event(s):



,event type,eligible balance,status
0,ExpiryEvent,"10,000,000.00",Active


## 6b. The transactions the expiry event carries

Now that `Expiry` is registered from section 3, `ApplicableInstrumentEvent.transactions` on the
matching `ExpiryEvent` comes back populated.

In [9]:
rows = []
for ev in applicable:
    if ev.instrument_event_type != "ExpiryEvent":
        continue
    for txn in (ev.transactions or []):
        rows.append({
            "event": ev.instrument_event_type,
            "txn type": getattr(txn, "type", None),
            "units": getattr(txn, "units", None),
            "transaction date": getattr(txn, "transaction_date", None),
            "settlement date": getattr(txn, "settlement_date", None),
        })

if rows:
    display(pd.DataFrame(rows))
    print("The registered 'Expiry' transaction type closes the swaption position at its own "
          "expiry.")
else:
    print("No forecast transactions. Check the transaction type registered in section 3.")

,event,txn type,units,transaction date,settlement date
0,ExpiryEvent,Expiry,"10,000,000.00",2026-03-03 00:00:00+00:00,2026-03-03 00:00:00+00:00


The registered 'Expiry' transaction type closes the swaption position at its own expiry.


---
# Summary

1. A cross-currency swaption is just an `InterestRateSwaption` whose `swap` field carries the
   underlying `InterestRateSwap` inline, with legs sitting in two different currencies -- there's
   no separate instrument to create for the swap itself.
2. The option's real expiry is its `exercise_date`, and this notebook sets it explicitly rather
   than letting it default from the underlying swap's `start_date`. Value the position at or
   after that date and you get nothing back, silently.
3. Under `SimpleStatic` it's reported at a quoted mark -- the legs describe the instrument, but
   don't drive the number.
4. The applicable event is `ExpiryEvent`, not `MaturityEvent`: the option's lifecycle ends at
   expiry, and the forecasted `Expiry` transaction settles on that same date.
5. `ExpiryEvent`'s `.transactions` stay empty until a transaction type is registered. What you
   register is the generic `Expiry`, not an instrument-qualified alias like
   `InterestRateSwaptionExpiry` -- `Expiry` covers every instrument type `ExpiryEvent` applies to
   (swaptions, equity/exchange-traded/bond/CDS options, TBA options), and you only need to
   register it once, in `scope=default, source=default`.
6. Same as `instrumentEventConfiguration` on the portfolio in section 4: none of this fires unless
   the portfolio was created pointing at a recipe from the start -- it can't be bolted on later.

In [10]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instrument : {SWAPTION_LUID}")

Scope      : CrossCurrencySwaptionDemo
Portfolio  : CrossCurrencySwaptionDemo/cross-currency-swaption-demo-book
Recipe     : CrossCurrencySwaptionDemo/cross-currency-swaption-demo-recipe
Instrument : LUID_00003DFX
